In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
from sklearn.preprocessing import MinMaxScaler,LabelEncoder,OneHotEncoder,PolynomialFeatures
from sklearn.model_selection import train_test_split,cross_val_score,RandomizedSearchCV 
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import AdaBoostRegressor,GradientBoostingRegressor,RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error,root_mean_squared_error

In [6]:
import pickle

In [7]:
df=pd.read_csv("3social_media_vs_productivity.csv")

Basic Understanding

In [8]:
df.shape

(30000, 19)

In [9]:
df.columns

Index(['age', 'gender', 'job_type', 'daily_social_media_time',
       'social_platform_preference', 'number_of_notifications',
       'work_hours_per_day', 'perceived_productivity_score',
       'actual_productivity_score', 'stress_level', 'sleep_hours',
       'screen_time_before_sleep', 'breaks_during_work', 'uses_focus_apps',
       'has_digital_wellbeing_enabled', 'coffee_consumption_per_day',
       'days_feeling_burnout_per_month', 'weekly_offline_hours',
       'job_satisfaction_score'],
      dtype='object')

In [10]:
df.head()

,age,gender,job_type,daily_social_media_time,social_platform_preference,number_of_notifications,work_hours_per_day,perceived_productivity_score,actual_productivity_score,stress_level,sleep_hours,screen_time_before_sleep,breaks_during_work,uses_focus_apps,has_digital_wellbeing_enabled,coffee_consumption_per_day,days_feeling_burnout_per_month,weekly_offline_hours,job_satisfaction_score
0,56,Male,Unemployed,4.180940,Facebook,61,6.753558,8.040464,7.291555,4.0,5.116546,0.419102,8,False,False,4,11,21.927072,6.336688
1,46,Male,Health,3.249603,Twitter,59,9.169296,5.063368,5.165093,7.0,5.103897,0.671519,7,True,True,2,25,0.000000,3.412427
2,32,Male,Finance,NaN,Twitter,57,7.910952,3.861762,3.474053,4.0,8.583222,0.624378,0,True,False,3,17,10.322044,2.474944
3,60,Female,Unemployed,NaN,Facebook,59,6.355027,2.916331,1.774869,6.0,6.052984,1.204540,1,False,False,0,4,23.876616,1.733670
4,25,Male,IT,NaN,Telegram,66,6.214096,8.868753,NaN,7.0,5.405706,1.876254,1,False,True,1,30,10.653519,9.693060


In [11]:
df.tail()

,age,gender,job_type,daily_social_media_time,social_platform_preference,number_of_notifications,work_hours_per_day,perceived_productivity_score,actual_productivity_score,stress_level,sleep_hours,screen_time_before_sleep,breaks_during_work,uses_focus_apps,has_digital_wellbeing_enabled,coffee_consumption_per_day,days_feeling_burnout_per_month,weekly_offline_hours,job_satisfaction_score
29995,34,Female,Health,1.877297,Facebook,59,10.226358,3.348512,3.465815,8.0,5.480462,1.412655,9,False,False,4,5,21.776927,NaN
29996,39,Male,Health,4.437784,Instagram,46,4.692862,8.133213,6.659294,8.0,3.045393,0.148936,3,False,False,1,29,4.111370,6.155613
29997,42,Male,Education,17.724981,TikTok,64,10.915036,8.611005,8.658912,5.0,5.491520,1.224296,10,False,False,1,2,1.888315,6.285237
29998,20,Female,Education,3.796634,Instagram,56,6.937410,7.767076,6.895583,8.0,6.816069,0.234483,1,False,False,2,9,12.511871,7.854711
29999,44,Male,Unemployed,NaN,Twitter,70,8.069883,6.311227,5.402726,3.0,6.765248,0.993090,5,False,True,1,4,6.324954,7.388790


In [12]:
df['social_platform_preference'].unique()

array(['Facebook', 'Twitter', 'Telegram', 'TikTok', 'Instagram'],
      dtype=object)

In [13]:
df.describe()

,age,daily_social_media_time,number_of_notifications,work_hours_per_day,perceived_productivity_score,actual_productivity_score,stress_level,sleep_hours,screen_time_before_sleep,breaks_during_work,coffee_consumption_per_day,days_feeling_burnout_per_month,weekly_offline_hours,job_satisfaction_score
count,30000.000000,27235.000000,30000.000000,30000.000000,28386.000000,27635.000000,28096.000000,27402.000000,27789.000000,30000.000000,30000.000000,30000.000000,30000.000000,27270.000000
mean,41.486867,3.113418,59.958767,6.990792,5.510488,4.951805,5.514059,6.500247,1.025568,4.992200,1.999300,15.557067,10.360655,4.964901
std,13.835221,2.074813,7.723772,1.997736,2.023470,1.883378,2.866344,1.464004,0.653355,3.173737,1.410047,9.252956,7.280415,2.121194
min,18.000000,0.000000,30.000000,0.000000,2.000252,0.296812,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,30.000000,1.639566,55.000000,5.643771,3.757861,3.373284,3.000000,5.493536,0.528490,2.000000,1.000000,8.000000,4.541872,3.363580
50%,41.000000,3.025913,60.000000,6.990641,5.525005,4.951742,6.000000,6.498340,1.006159,5.000000,2.000000,16.000000,10.013677,4.951049
75%,53.000000,4.368917,65.000000,8.354725,7.265776,6.526342,8.000000,7.504143,1.477221,8.000000,3.000000,24.000000,15.300809,6.581323
max,65.000000,17.973256,90.000000,12.000000,8.999376,9.846258,10.000000,10.000000,3.000000,10.000000,10.000000,31.000000,40.964769,10.000000


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   age                             30000 non-null  int64  
 1   gender                          30000 non-null  object 
 2   job_type                        30000 non-null  object 
 3   daily_social_media_time         27235 non-null  float64
 4   social_platform_preference      30000 non-null  object 
 5   number_of_notifications         30000 non-null  int64  
 6   work_hours_per_day              30000 non-null  float64
 7   perceived_productivity_score    28386 non-null  float64
 8   actual_productivity_score       27635 non-null  float64
 9   stress_level                    28096 non-null  float64
 10  sleep_hours                     27402 non-null  float64
 11  screen_time_before_sleep        27789 non-null  float64
 12  breaks_during_work              

In [15]:
df.duplicated().sum()

np.int64(0)

In [16]:
df.isna().sum()

age                                  0
gender                               0
job_type                             0
daily_social_media_time           2765
social_platform_preference           0
number_of_notifications              0
work_hours_per_day                   0
perceived_productivity_score      1614
actual_productivity_score         2365
stress_level                      1904
sleep_hours                       2598
screen_time_before_sleep          2211
breaks_during_work                   0
uses_focus_apps                      0
has_digital_wellbeing_enabled        0
coffee_consumption_per_day           0
days_feeling_burnout_per_month       0
weekly_offline_hours                 0
job_satisfaction_score            2730
dtype: int64

In [17]:
((df.shape[0]-df.dropna().shape[0])/df.shape[0])*100

43.086666666666666

Univariate Analysis

In [18]:
col_tocheck_valuecounts=['gender', 'job_type','social_platform_preference','uses_focus_apps',
       'has_digital_wellbeing_enabled']
for col in col_tocheck_valuecounts:
    a=df[col].value_counts()
    print(a)
    print('_'*20,end='\n\n')

gender
Male      14452
Female    14370
Other      1178
Name: count, dtype: int64
____________________

job_type
Education     5055
IT            5026
Finance       5017
Student       5012
Unemployed    4958
Health        4932
Name: count, dtype: int64
____________________

social_platform_preference
TikTok       6096
Telegram     6013
Instagram    6006
Twitter      5964
Facebook     5921
Name: count, dtype: int64
____________________

uses_focus_apps
False    20979
True      9021
Name: count, dtype: int64
____________________

has_digital_wellbeing_enabled
False    22602
True      7398
Name: count, dtype: int64
____________________



In [19]:
df['number_of_notifications'].mean()

np.float64(59.95876666666667)

In [20]:
df['age'].median()

np.float64(41.0)

In [21]:
df['job_type'].mode()

0    Education
Name: job_type, dtype: object

In [22]:
(df['sleep_hours']<6).sum()

np.int64(10103)

In [23]:
df['stress_level'].nunique()

10

In [24]:
(df['days_feeling_burnout_per_month']>10).sum()

np.int64(19747)

In [25]:
df['weekly_offline_hours'].mean()

np.float64(10.360654646637764)

In [26]:
(df['daily_social_media_time']>5).sum()

np.int64(4383)

In [27]:
df.columns

Index(['age', 'gender', 'job_type', 'daily_social_media_time',
       'social_platform_preference', 'number_of_notifications',
       'work_hours_per_day', 'perceived_productivity_score',
       'actual_productivity_score', 'stress_level', 'sleep_hours',
       'screen_time_before_sleep', 'breaks_during_work', 'uses_focus_apps',
       'has_digital_wellbeing_enabled', 'coffee_consumption_per_day',
       'days_feeling_burnout_per_month', 'weekly_offline_hours',
       'job_satisfaction_score'],
      dtype='object')

In [28]:
col_to_fill=['daily_social_media_time','perceived_productivity_score',
       'actual_productivity_score', 'stress_level', 'sleep_hours',
       'screen_time_before_sleep','job_satisfaction_score']
for col in col_to_fill:
    df[col]=df[col].fillna(df[col].median())

multivariate analysis

In [29]:
df[['daily_social_media_time','actual_productivity_score']].corr()

,daily_social_media_time,actual_productivity_score
daily_social_media_time,1.000000,-0.010072
actual_productivity_score,-0.010072,1.000000


In [30]:
df[['stress_level','actual_productivity_score']].corr()

,stress_level,actual_productivity_score
stress_level,1.00000,0.00095
actual_productivity_score,0.00095,1.00000


In [31]:
df.groupby('gender')['actual_productivity_score'].mean()

gender
Female    4.957014
Male      4.954793
Other     4.851485
Name: actual_productivity_score, dtype: float64

In [32]:
df.groupby('job_type')['actual_productivity_score'].mean()

job_type
Education     4.927235
Finance       4.964768
Health        4.925115
IT            5.002925
Student       4.908015
Unemployed    4.982703
Name: actual_productivity_score, dtype: float64

In [33]:
df[['stress_level', 'sleep_hours']].corr()

,stress_level,sleep_hours
stress_level,1.000000,0.003947
sleep_hours,0.003947,1.000000


In [34]:
df.groupby('uses_focus_apps')['stress_level'].mean()

uses_focus_apps
False    5.540493
True     5.555149
Name: stress_level, dtype: float64

In [35]:
df[['number_of_notifications','stress_level']].corr()

,number_of_notifications,stress_level
number_of_notifications,1.000000,-0.003459
stress_level,-0.003459,1.000000


In [36]:
df.groupby('social_platform_preference')['actual_productivity_score'].mean()

social_platform_preference
Facebook     4.929769
Instagram    4.954714
Telegram     4.926207
TikTok       4.976946
Twitter      4.970837
Name: actual_productivity_score, dtype: float64

In [37]:
df[['actual_productivity_score','weekly_offline_hours']].corr()

,actual_productivity_score,weekly_offline_hours
actual_productivity_score,1.000000,-0.004632
weekly_offline_hours,-0.004632,1.000000


In [38]:
df.groupby('job_type')['days_feeling_burnout_per_month'].mean()

job_type
Education     15.407517
Finance       15.666733
Health        15.538321
IT            15.662356
Student       15.484038
Unemployed    15.584308
Name: days_feeling_burnout_per_month, dtype: float64

In [ ]:
df.groupby(['gender','job_type'])['actual_productivity_score'].mean().sort_values()

In [40]:
df.groupby(['uses_focus_apps','has_digital_wellbeing_enabled'])['actual_productivity_score'].mean()

uses_focus_apps  has_digital_wellbeing_enabled
False            False                            4.956431
                 True                             4.942318
True             False                            4.963697
                 True                             4.904990
Name: actual_productivity_score, dtype: float64

In [ ]:
df.groupby(['job_type','social_platform_preference'])['stress_level'].mean()

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(df.corr(numeric_only=True),annot=True,fmt=".2f",linewidths=.5)
plt.show()

In [ ]:
plt.hist(df['age'],bins=10,edgecolor='k')
plt.title('distribution of age')
plt.xlabel('age')
plt.ylabel('count')
plt.xticks(range(16,70,2),rotation=90)
plt.show()

In [ ]:
plt.hist(df['actual_productivity_score'],bins=8,edgecolor='k')
plt.title('distribution of actual productivity')
plt.xlabel('productivity score')
plt.ylabel('count')
plt.xticks(range(0,10))
plt.show()

In [ ]:
df['gender'].value_counts().plot(kind='pie',autopct="%1.1f%%")
plt.title('gender distibution')
plt.show()

In [ ]:
df['uses_focus_apps'].value_counts().plot(kind='pie',autopct="%d%%")
plt.title('focus app usage')
plt.show()

In [ ]:
df['has_digital_wellbeing_enabled'].value_counts().plot(kind='pie',autopct="%d%%")
plt.title('digital welbieng enabled')
plt.show()

In [ ]:
df['job_type'].value_counts().plot(kind='bar')
plt.xlabel('job type')
plt.ylabel('count')
plt.show()

In [ ]:
df['social_platform_preference'].value_counts().plot(kind='bar')
plt.xlabel('platform')
plt.ylabel('count')
plt.title('social media platform preference')
plt.show()

In [ ]:
df.groupby('job_type')['actual_productivity_score'].mean().plot(kind='bar')
plt.xlabel('job type')
plt.ylabel('average productivity score')
plt.title('average productivity by job type')
plt.show()

In [ ]:
df.groupby('uses_focus_apps')['stress_level'].mean().plot(kind='bar')
plt.xlabel('Uses Focus Apps')
plt.ylabel('Average Stress Level')
plt.title('Stress Level by Focus App Usage')
plt.show()

In [ ]:
col_tocheck_boxplot=['age','daily_social_media_time','number_of_notifications',
       'work_hours_per_day', 'perceived_productivity_score',
       'actual_productivity_score', 'stress_level', 'sleep_hours',
       'screen_time_before_sleep', 'breaks_during_work', 'coffee_consumption_per_day',
       'days_feeling_burnout_per_month', 'weekly_offline_hours',
       'job_satisfaction_score']
for col in col_tocheck_boxplot:
    plt.boxplot(df[col])
    plt.title(col)
    plt.show()

In [53]:
df.loc[df['coffee_consumption_per_day']>6]

,age,gender,job_type,daily_social_media_time,social_platform_preference,number_of_notifications,work_hours_per_day,perceived_productivity_score,actual_productivity_score,stress_level,sleep_hours,screen_time_before_sleep,breaks_during_work,uses_focus_apps,has_digital_wellbeing_enabled,coffee_consumption_per_day,days_feeling_burnout_per_month,weekly_offline_hours,job_satisfaction_score
99,18,Male,Unemployed,3.464868,Instagram,62,6.648502,8.487270,7.788354,8.0,5.140200,1.448916,3,False,False,7,16,1.892515,4.951049
608,40,Female,Student,2.614267,Twitter,58,3.890973,8.904853,7.929358,2.0,5.562917,1.647898,6,False,False,7,15,9.065356,8.019669
901,19,Male,IT,0.000000,Twitter,64,11.517955,5.767177,5.749209,3.0,6.498340,1.159038,1,False,True,7,11,11.725778,5.952772
946,61,Female,Finance,3.255492,Instagram,49,6.298203,6.177558,4.951742,3.0,6.164833,1.674702,8,True,False,7,12,5.479297,4.601104
1154,29,Female,IT,1.596466,Facebook,59,3.397056,6.364936,6.149528,6.0,6.239752,1.339053,4,False,True,7,4,0.000000,6.186559
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28159,44,Male,Health,0.000000,Telegram,55,6.608266,2.915776,4.951742,10.0,5.865578,1.120325,5,False,False,8,30,14.437043,2.812137
28879,32,Other,IT,4.551644,Twitter,64,7.382231,6.311715,5.987833,3.0,3.354182,1.006159,8,False,False,7,30,9.205720,5.238091
29263,28,Female,Finance,5.109316,Instagram,56,3.349012,4.046453,4.268999,5.0,5.222530,1.204144,0,True,False,8,2,0.322100,5.240538
29619,45,Female,Student,3.025913,Twitter,59,5.289151,5.116967,4.457814,1.0,6.498340,1.210316,0,True,True,7,6,6.934792,5.335318


In [54]:
df.head()

,age,gender,job_type,daily_social_media_time,social_platform_preference,number_of_notifications,work_hours_per_day,perceived_productivity_score,actual_productivity_score,stress_level,sleep_hours,screen_time_before_sleep,breaks_during_work,uses_focus_apps,has_digital_wellbeing_enabled,coffee_consumption_per_day,days_feeling_burnout_per_month,weekly_offline_hours,job_satisfaction_score
0,56,Male,Unemployed,4.180940,Facebook,61,6.753558,8.040464,7.291555,4.0,5.116546,0.419102,8,False,False,4,11,21.927072,6.336688
1,46,Male,Health,3.249603,Twitter,59,9.169296,5.063368,5.165093,7.0,5.103897,0.671519,7,True,True,2,25,0.000000,3.412427
2,32,Male,Finance,3.025913,Twitter,57,7.910952,3.861762,3.474053,4.0,8.583222,0.624378,0,True,False,3,17,10.322044,2.474944
3,60,Female,Unemployed,3.025913,Facebook,59,6.355027,2.916331,1.774869,6.0,6.052984,1.204540,1,False,False,0,4,23.876616,1.733670
4,25,Male,IT,3.025913,Telegram,66,6.214096,8.868753,4.951742,7.0,5.405706,1.876254,1,False,True,1,30,10.653519,9.693060


model evaluation

In [55]:
x=df.drop(columns=['actual_productivity_score','perceived_productivity_score'])
y=df['actual_productivity_score']

In [56]:
new_columns=['age', 'daily_social_media_time','number_of_notifications',
       'work_hours_per_day', 'stress_level', 'sleep_hours',
       'screen_time_before_sleep', 'breaks_during_work', 'coffee_consumption_per_day',
       'days_feeling_burnout_per_month', 'weekly_offline_hours',
       'job_satisfaction_score','uses_focus_apps',
       'has_digital_wellbeing_enabled', 'gender', 'job_type','social_platform_preference']
x=x[new_columns]

In [57]:
x['uses_focus_apps'] = x['uses_focus_apps'].astype(int)
x['has_digital_wellbeing_enabled'] = x['has_digital_wellbeing_enabled'].astype(int)

In [58]:
onehot=OneHotEncoder(sparse_output=False,drop='first')
onehot.fit(df[['gender', 'job_type','social_platform_preference']])

,categories,'auto'
,drop,'first'
,sparse_output,False
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [59]:
x.columns

Index(['age', 'daily_social_media_time', 'number_of_notifications',
       'work_hours_per_day', 'stress_level', 'sleep_hours',
       'screen_time_before_sleep', 'breaks_during_work',
       'coffee_consumption_per_day', 'days_feeling_burnout_per_month',
       'weekly_offline_hours', 'job_satisfaction_score', 'uses_focus_apps',
       'has_digital_wellbeing_enabled', 'gender', 'job_type',
       'social_platform_preference'],
      dtype='object')

In [60]:
x=pd.get_dummies(x,columns=['gender', 'job_type','social_platform_preference'],drop_first=True,dtype=int)

In [61]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=42)

In [62]:
scaler=MinMaxScaler()
x_train_scaler=scaler.fit_transform(x_train)
x_test_scaler=scaler.transform(x_test)

In [ ]:
models=[KNeighborsRegressor(),DecisionTreeRegressor(),LinearRegression() ,
XGBRegressor(),AdaBoostRegressor(),GradientBoostingRegressor(),RandomForestRegressor(),SVR()]
models=[LinearRegression() ,XGBRegressor(),GradientBoostingRegressor()]
for model in models:
    print(str(model).split('(')[0])
    model.fit(x_train_scaler,y_train)
    y_pred=model.predict(x_test_scaler)
    print("r2_score",r2_score(y_test,y_pred))
    print("mse",mean_squared_error(y_test,y_pred))
    print("rmse",root_mean_squared_error(y_test,y_pred))
    print("mae",mean_absolute_error(y_test,y_pred))
    print('train_score',model.score(x_train_scaler,y_train))
    print('test_score',model.score(x_test_scaler,y_test))
    print("-"*20)

In [ ]:
models=[LinearRegression() ,XGBRegressor(),GradientBoostingRegressor()]
for model in models: 
    print(str(model).split('(')[0])
    print(cross_val_score(model,x,y,cv=5).mean())
    print('-'*20)

In [65]:
final_model=GradientBoostingRegressor()
final_model.fit(x_train_scaler,y_train)
y_pred=final_model.predict(x_test_scaler)

In [66]:
data={'model':final_model,'scaler':scaler,'onehot':onehot}
with open('socialmedia-productivity.pkl','wb') as obj1:
    pickle.dump(data,obj1)